# EDGAR Financial Sentiment — Part 6: Synthetic Data Generation

**The Track B → Track A bridge.** In Part 5 you used Mistral to *classify*. Here you use the same model to *manufacture labeled training data*, add it to a small real training set, and **measure** whether it helps a downstream classifier.

**What you'll practice (the core concept):** prompting an LLM to **generate labeled examples** (`build_generation_prompt`) and **parsing** the generated text into clean `(sentence, label)` pairs (`parse_generated`) — those are your blanks. Everything else (the generation driver and the downstream bert-base classifier that measures the effect) is provided.

> **Run in Google Colab with a T4 GPU.** Uses the gated Mistral model (Part 4 login) for generation, then a small bert-base classifier to measure the payoff.

## 0. Why generate data? The labeling bottleneck

Fine-tuning (Parts 2–4) needs **labeled** data, and the labels are the expensive part. Financial PhraseBank exists only because annotators hand-labeled thousands of sentences. What if you *don't* have that — a new domain, a new task, no budget for labeling?

**Track B's trick:** ask an LLM to write the examples *for* you. Prompt it for "10 short financial sentences with negative sentiment," and every sentence comes **pre-labeled** — you asked for negative, so the label is negative. Add a few hundred of those to a small real training set and you've expanded your labels almost for free. The same model that *classified* in Part 5 now *generates training data* here.

**The honest question this notebook measures:** *does it actually help?* Synthetic data should help most when real labels are **scarce**, so we simulate that — a tiny real seed set (a few dozen examples), augmented with LLM-generated sentences — and we measure the accuracy change on the **real** test set. Sometimes it lifts accuracy; sometimes synthetic noise (templated or mislabeled sentences) hurts. We find out by **measuring**, not assuming.

**The experiment:**
1. Take a *small* real labeled seed set (simulating scarcity).
2. Prompt Mistral to generate labeled synthetic sentences. *(your code)*
3. Train a bert-base classifier on **real-only** vs. **real + synthetic**.
4. Compare test accuracy — the delta is what the synthetic data bought.

## 1. Setup

In [1]:
!pip install -q -U transformers bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.8 MB/s eta 0:00:00


In [2]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # reduce fragmentation (Part 4 lesson)

import random, re, gc, io, zipfile, requests
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from datasets import Dataset as HFDataset, ClassLabel

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

Device: cuda


### Hugging Face login (Mistral is gated)
Same as Parts 4–5: accept the license, make a token, log in.

In [3]:
from huggingface_hub import login
login()   # paste your HF token when prompted

## 2. Config & seeds

In [5]:
torch.manual_seed(10); random.seed(10); np.random.seed(10)

MODEL_ID           = 'mistralai/Mistral-7B-Instruct-v0.3'
LABELS             = ['negative', 'neutral', 'positive']

SEED_PER_CLASS     = 20    # tiny REAL training set per class (simulates scarce labels: 60 total)
SYNTH_PER_CLASS    = 60    # synthetic sentences to generate per class
SENTENCES_PER_CALL = 12    # how many to ask for in a single generation call
GEN_TEMPERATURE    = 0.9   # sampling temperature -> diversity (greedy would repeat itself)

## 3. Real data — a *small* seed set + the real test set

Same PhraseBank + split as before, but we deliberately keep only `SEED_PER_CLASS` real examples per class for training (the scarcity we're simulating). The test set stays full and **real** — that's what we measure on.

In [6]:
_URL = 'https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip'
_z = zipfile.ZipFile(io.BytesIO(requests.get(_URL, timeout=120).content))
_member = next(n for n in _z.namelist() if n.endswith('Sentences_AllAgree.txt'))
_raw = _z.read(_member).decode('latin-1')
_label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
_rows = []
for _line in _raw.splitlines():
    _line = _line.strip()
    if not _line:
        continue
    _sent, _lab = _line.rsplit('@', 1)
    _rows.append({'sentence': _sent, 'label': _label_map[_lab.strip()]})
_df = pd.DataFrame(_rows)
pb_full = HFDataset.from_pandas(_df).cast_column('label', ClassLabel(names=LABELS))
pb = pb_full.train_test_split(test_size=0.2, seed=10, stratify_by_column='label')
pb_train, pb_test = pb['train'], pb['test']

# tiny REAL seed set: SEED_PER_CLASS examples per class
seed_examples = []
for _cls in range(3):
    _exs = [e for e in pb_train if e['label'] == _cls][:SEED_PER_CLASS]
    seed_examples.extend((e['sentence'], _cls) for e in _exs)
random.shuffle(seed_examples)
print('Real seed examples:', len(seed_examples), '| Real test examples:', len(pb_test))

Casting the dataset:   0%|          | 0/2264 [00:00<?, ? examples/s]

Real seed examples: 60 | Real test examples: 453


## 4. Load Mistral-7B (4-bit) for generation  *(provided)*

Loaded as a causal LM, same as Part 5. Note `generate` uses **sampling** (`do_sample=True`, temperature) — unlike Part 5's greedy decoding — because we want *diverse* sentences, not the same one every call.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map={'': 0})
model.eval()
print('Loaded', MODEL_ID)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded mistralai/Mistral-7B-Instruct-v0.3


In [17]:
@torch.no_grad()
def generate(prompt_text, max_new_tokens=320, temperature=GEN_TEMPERATURE, top_p=0.95):
    """Sample a reply from the model (sampling -> diverse generations)."""
    messages = [{'role': 'user', 'content': prompt_text}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(device)
    out = model.generate(inputs=inputs['input_ids'], attention_mask=inputs['attention_mask'], max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=temperature, top_p=top_p, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

## 5. Write the generation prompt  &larr; **your code**

This is the heart of synthetic-data generation. Write `build_generation_prompt(label_name, n)` that asks the model for `n` short financial sentences expressing a **given** sentiment. Good generation prompts:
- state the **sentiment** clearly (the label you'll attach to every sentence it returns);
- ask for **variety** (different companies, sectors, wording) so the data isn't templated;
- demand a **clean format** you can parse — a numbered list, one sentence per line, no commentary.

The format you ask for here determines how easy `parse_generated` is next — design them together.

In [27]:
def build_generation_prompt(label_name, n):
    """Ask the model for n short financial sentences with the given sentiment."""
    ### YOUR CODE HERE
    # Return a single instruction string that:
    #  - asks for `n` short one-sentence financial-news statements
    #  - tells the model the sentiment must be `label_name` (negative/neutral/positive)
    #  - asks for variety (different companies/sectors/wording)
    #  - demands a clean NUMBERED LIST, one sentence per line, no commentary
    #    (the format you choose here is what parse_generated must read)
    instructions = [
      "You are a financial writer and your task is to write 1 sentence that expresses explicityly positive, neutral, or negative sentiment about a company.",
      "Write as if you are the management of the company discussing the financial health and outlook.",
      "Do not repeat any company or industry, and do not use jargon.",
      f"Generate {n} sentences with {label_name} sentiment.",
      f"Format your response as a numbers list separated by new lines: \n1. [sentence]"
    ]
    return '\n'.join(instructions)
    ### END YOUR CODE

### See a generation prompt + a real reply
Run after filling in `build_generation_prompt`. Look at the raw reply — its format is what your parser must handle.

In [28]:
_demo = build_generation_prompt('negative', 5)
print('=== PROMPT ===')
print(_demo)
print('\n=== MODEL REPLY ===')
print(generate(_demo))

=== PROMPT ===
You are a financial writer and your task is to write 1 sentence that expresses explicityly positive, neutral, or negative sentiment about a company.
Write as if you are the management of the company discussing the financial health and outlook.
Do not repeat any company or industry, and do not use jargon.
Generate 5 sentences with negative sentiment.
Format your response as a numbers list separated by new lines: 
1. [sentence]

=== MODEL REPLY ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


1. "Despite our best efforts, we've experienced a significant decrease in revenue this quarter, primarily due to the economic downturn and increased competition in our industry."

2. "Our cost structure has become increasingly burdensome, with operational expenses outpacing revenue growth, resulting in a challenging financial outlook moving forward."

3. "We regret to inform you that our earnings this quarter have fallen short of expectations, reflecting a drop in demand for our products and services."

4. "The ongoing global supply chain disruptions have had a detrimental impact on our production capacity, leading to decreased inventory levels and missed delivery deadlines."

5. "Unfortunately, our recent expansion into new markets has not been as successful as anticipated, leading to higher-than-expected costs and slower growth than we had projected."


## 6. Parse the generated sentences  &larr; **your code**

`parse_generated(text)` turns the model's reply (a numbered list) into a clean **list of sentence strings**. For each line: keep only lines that look like `1. ...` / `2) ...`, strip the numbering and surrounding quotes, and drop anything too short to be a real sentence. (A regex like `^\d+[.)]\s*(.+)$` matches a leading number.)

In [32]:
import re

def parse_generated(text):
    """Pull individual sentences out of the model's numbered-list reply."""
    ### YOUR CODE HERE
    # 1. split text into lines
    # 2. keep lines that start with a number (regex r'^\d+[.)]\s*(.+)$' captures the sentence)
    # 3. strip the numbering + surrounding quotes; drop anything <= ~10 chars
    # 4. return the list of cleaned sentence strings
    sentences = text.split("\n")
    cleaned = []
    for sentence in sentences:
      s_match = re.match(r'^\d+[.)]\s*(.+)$', sentence)
      if s_match:
        cleaned_sentence = s_match.group(1).strip('"')
        cleaned_sentence = cleaned_sentence.strip()
        if bool(re.match(r"^(?=(?:\s*\S){11,})", cleaned_sentence)):
          cleaned.append(cleaned_sentence)
    return cleaned
    ### END YOUR CODE

### Test your parser

In [33]:
_sample = '1. Acme Corp posted a steep quarterly loss.\n2) Beta Inc cut its full-year guidance.\nHere you go:\n3. Gamma Ltd missed revenue estimates.'
print(parse_generated(_sample))   # expect 3 clean sentences, the 'Here you go:' line dropped

['Acme Corp posted a steep quarterly loss.', 'Beta Inc cut its full-year guidance.', 'Gamma Ltd missed revenue estimates.']


## 7. Generate the synthetic dataset  *(provided driver — uses your two functions)*

Loops over the three classes, calls your prompt + parser until it has `SYNTH_PER_CLASS` sentences each, and labels every sentence with the class it asked for. This is the slow cell (many generations) — a few minutes.

In [34]:
def generate_synthetic(per_class, per_call=SENTENCES_PER_CALL, max_calls=25):
    synth = []
    for cls, name in enumerate(LABELS):
        collected, calls = [], 0
        while len(collected) < per_class and calls < max_calls:
            collected.extend(parse_generated(generate(build_generation_prompt(name, per_call))))
            calls += 1
        synth.extend((s, cls) for s in collected[:per_class])
        print(f'  {name}: collected {len(collected)} -> kept {min(len(collected), per_class)}')
    return synth

print('Generating synthetic data (this takes a few minutes)...')
SYNTH = generate_synthetic(SYNTH_PER_CLASS)
print('Total synthetic (raw):', len(SYNTH))

Generating synthetic data (this takes a few minutes)...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  negative: collected 69 -> kept 60
  neutral: collected 69 -> kept 60
  positive: collected 63 -> kept 60
Total synthetic (raw): 180


## 8. Inspect quality — dedup, balance, eyeball  *(instrument)*

Synthetic data is only as good as the generator. Before trusting it: remove any sentence that duplicates a real seed/test sentence (leakage) or repeats another synthetic one, check the class balance, and **read a few of them**.

In [35]:
real_texts = {s.lower().strip() for s, _ in seed_examples}
real_texts |= {e['sentence'].lower().strip() for e in pb_test}

seen, synth_clean = set(), []
for s, c in SYNTH:
    k = s.lower().strip()
    if k in real_texts or k in seen:
        continue
    seen.add(k); synth_clean.append((s, c))

print(f'synthetic: {len(SYNTH)} raw -> {len(synth_clean)} after dedup')
print('class balance:', dict(Counter(LABELS[c] for _, c in synth_clean)))
print('\nA few generated examples:')
for name in LABELS:
    for s, c in synth_clean:
        if LABELS[c] == name:
            print(f'  [{name}] {s}')
            break

synthetic: 180 raw -> 180 after dedup
class balance: {'negative': 60, 'neutral': 60, 'positive': 60}

A few generated examples:
  [negative] Despite our best efforts, we've seen a significant decline in our quarterly revenue, a trend that's concerning and requires immediate attention.
  [neutral] Our quarterly revenue growth remains steady, reflecting our consistent focus on expanding customer base and strengthening existing relationships.
  [positive] We're thrilled to report that our sales have increased by 25% over the past quarter, reflecting a growing demand for our innovative products.


#### ✍️ Reflect — is the synthetic data any good?
Read the examples and answer:
- Do they look like real financial-news sentences? _…_
- Does the label look correct for each — especially the **neutral** ones? _…_
- Are they varied, or templated/repetitive? How many were dropped as duplicates? _…_
- Predict: will adding these help, do nothing, or hurt the classifier? _…_

## 9. Free Mistral, set up the downstream classifier  *(provided)*

We measure the effect with a small, fast **bert-base `[CLS]` classifier** (Part 3) — not the 7B — so we free Mistral from the GPU first. `train_and_eval` fine-tunes a fresh classifier on whatever examples you pass and returns test accuracy on the **real** test set.

In [36]:
del model
gc.collect(); torch.cuda.empty_cache()
print(f'Freed Mistral. GPU allocated now: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

Freed Mistral. GPU allocated now: 3.86 GB


In [37]:
def train_and_eval(train_examples, epochs=4, lr=2e-5, bs=16, seed=10):
    """Fine-tune a fresh bert-base [CLS] classifier on train_examples [(sentence,label), ...],
    evaluate on the REAL PhraseBank test set, return test accuracy %."""
    torch.manual_seed(seed)
    btok = AutoTokenizer.from_pretrained('bert-base-cased')
    enc = AutoModel.from_pretrained('bert-base-cased').to(device)

    class _DS(Dataset):
        def __init__(self, examples):
            self.rows = []
            for sent, lab in examples:
                e = btok(sent, max_length=64, truncation=True, padding='max_length', return_tensors='pt')
                self.rows.append({'input_ids': e['input_ids'].squeeze(0),
                                  'attention_mask': e['attention_mask'].squeeze(0),
                                  'label': torch.tensor(lab, dtype=torch.int64)})
        def __len__(self): return len(self.rows)
        def __getitem__(self, i): return self.rows[i]

    class _Clf(nn.Module):
        def __init__(self, encoder):
            super().__init__(); self.encoder = encoder
            self.head = nn.Linear(encoder.config.hidden_size, 3)
        def forward(self, b):
            out = self.encoder(input_ids=b['input_ids'], attention_mask=b['attention_mask'])
            return self.head(out.last_hidden_state[:, 0, :])

    test_examples = [(e['sentence'], e['label']) for e in pb_test]
    tr = DataLoader(_DS(train_examples), batch_size=bs, shuffle=True)
    te = DataLoader(_DS(test_examples), batch_size=32, shuffle=False)
    clf = _Clf(enc).to(device); opt = torch.optim.AdamW(clf.parameters(), lr=lr); lf = nn.CrossEntropyLoss()

    clf.train()
    for _ in range(epochs):
        for b in tr:
            b = {k: v.to(device) for k, v in b.items()}
            opt.zero_grad(); lf(clf(b), b['label']).backward(); opt.step()

    clf.eval(); correct = total = 0
    with torch.no_grad():
        for b in te:
            b = {k: v.to(device) for k, v in b.items()}
            correct += (clf(b).argmax(-1) == b['label']).sum().item(); total += b['label'].size(0)
    acc = 100 * correct / total
    del clf, enc; gc.collect(); torch.cuda.empty_cache()
    return acc

## 10. Measure the effect — real-only vs. real + synthetic

#### ✍️ Predict before you run
- Real-only accuracy guess: _…%_
- Real + synthetic guess: _…%_  (higher, same, or lower?)
- What would make synthetic data *hurt* here? _…_

In [38]:
print(f'Training on REAL seed only ({len(seed_examples)} examples)...')
acc_real = train_and_eval(seed_examples)
print(f'  -> {acc_real:.1f}%')
print(f'Training on REAL + SYNTHETIC ({len(seed_examples) + len(synth_clean)} examples)...')
acc_aug = train_and_eval(seed_examples + synth_clean)
print(f'  -> {acc_aug:.1f}%')

print('\n================ RESULT ================')
print(f'Real only:        {acc_real:.1f}%')
print(f'Real + synthetic: {acc_aug:.1f}%')
print(f'Synthetic lift:   {acc_aug - acc_real:+.1f} pts')

Training on REAL seed only (60 examples)...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  -> 71.7%
Training on REAL + SYNTHETIC (240 examples)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  -> 75.1%

================ RESULT ================
Real only:        71.7%
Real + synthetic: 75.1%
Synthetic lift:   +3.3 pts


## 11. Reflect — when does synthetic data help?

#### ✍️ Your turn
- Did synthetic data help, do nothing, or hurt? By how much? _…_
- Why does it help more with a *small* real seed than a large one? (Try re-running with the full `pb_train` as the seed.) _…_
- What single change to `build_generation_prompt` would most improve the synthetic data's quality? _…_
- Where does this fit among the methods in the use-case lab — when would you reach for synthetic data? _…_

## 12. Recap & next

You turned a *generator* into a *data factory*: a prompt that emits labeled sentences, a parser to clean them, a quality/dedup pass, and a measured before/after on a real test set. The lesson isn't "synthetic data is good" — it's that it's a **scarcity tool whose value you must measure**, capped by how good your generation prompt is.

**Next:** Part 7 — **structured extraction**: prompt the LLM to emit **JSON** (sentiment + guidance direction + key figures) from real press-release text, with chain-of-thought rationales — the on-ramp to real 8-K data.